# Biohub - Cell Tracking During Development
## Score: 0.872

## Configuration

In [ ]:
import os
from pathlib import Path

COMP_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development"
TEST_DIR = f"{COMP_DIR}/test"
ARTIFACTS_ROOT = Path(
    "/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts"
)
WEIGHTS_CANDIDATES = [
    Path("/kaggle/input/datasets/hongdaekim/biohub-350ep-checkpoint-pin-v1"),
    Path("/kaggle/input/hongdaekim/biohub-350ep-checkpoint-pin-v1"),
    Path("/kaggle/input/biohub-350ep-checkpoint-pin-v1"),
    Path(
        "/kaggle/input/datasets/hongdaekim/"
        "biohub-350ep-checkpoint-pin-v1/biohub-350ep-checkpoint-pin-v1"
    ),
]
REPO_DIR = "/kaggle/working/repo"
METHOD = "unet_transformer"
WEIGHTS = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
DET_THRESHOLD = 0.99
UNET_BATCH_SIZE = 4
USE_ILP = True
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.1
ILP_DISAPPEARANCE_WEIGHT = 0.1
ILP_DIVISION_WEIGHT = 1.0
OUTPUT_PATH = "/kaggle/working/submission.csv"

if (ARTIFACTS_ROOT / "repo").exists():
    ARTIFACTS = ARTIFACTS_ROOT
elif (ARTIFACTS_ROOT / "cellmot-baseline-artifacts" / "repo").exists():
    ARTIFACTS = ARTIFACTS_ROOT / "cellmot-baseline-artifacts"
else:
    ARTIFACTS = ARTIFACTS_ROOT

WEIGHTS_SRC = next(
    (
        path
        for path in WEIGHTS_CANDIDATES
        if (path / "edge_predictor_best.pth").exists()
    ),
    None,
)
if WEIGHTS_SRC is None:
    for path in Path("/kaggle/input").rglob("edge_predictor_best.pth"):
        if "thibautgoldsborough" in str(path):
            continue
        WEIGHTS_SRC = path.parent
        break

COMP_DIR, ARTIFACTS, WEIGHTS_SRC, OUTPUT_PATH


## Offline Install

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

assert WEIGHTS_SRC is not None, (
    "350ep checkpoint not found under /kaggle/input. "
    "Attach hongdaekim/biohub-350ep-checkpoint-pin-v1 and re-run."
)
assert (ARTIFACTS / "wheels").exists(), f"Missing wheels at {ARTIFACTS / 'wheels'}"
assert (ARTIFACTS / "repo").exists(), f"Missing repo at {ARTIFACTS / 'repo'}"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-index",
        "--find-links",
        str(ARTIFACTS / "wheels"),
        "--upgrade-strategy",
        "only-if-needed",
        "tracksdata",
        "zarr>=3.0.10",
        "pyscipopt",
    ],
    check=True,
)

shutil.copytree(ARTIFACTS / "repo", REPO_DIR, dirs_exist_ok=True)
if (ARTIFACTS / "weights").exists():
    shutil.copytree(ARTIFACTS / "weights", Path(REPO_DIR) / "weights", dirs_exist_ok=True)

weights_dir = Path(REPO_DIR) / "weights" / METHOD / "split_0"
weights_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(WEIGHTS_SRC / "edge_predictor_best.pth", weights_dir / "edge_predictor_best.pth")
config_src = WEIGHTS_SRC / "config.json"
if config_src.exists():
    shutil.copy2(config_src, weights_dir / "config.json")

sys.path.insert(0, f"{REPO_DIR}/src")
WEIGHTS_SRC, sorted(weights_dir.iterdir()), (weights_dir / "edge_predictor_best.pth").stat().st_size


## Test Split

In [ ]:
import json
from pathlib import Path

test_stems = sorted(
    path.name.replace(".zarr", "")
    for path in Path(TEST_DIR).glob("*.zarr")
)
splits_path = Path(REPO_DIR) / "kaggle_test_splits.json"
splits_path.write_text(
    json.dumps([{"split": 0, "train": [], "test": test_stems}])
)
len(test_stems), test_stems[:5]

## Inference

In [ ]:
import os
import subprocess

cmd = [
    sys.executable,
    "scripts/predict_unet_transformer.py",
    "--data-dir",
    TEST_DIR,
    "--splits",
    "kaggle_test_splits.json",
    "--split",
    "0",
    "--weights",
    WEIGHTS,
    "--unet-batch-size",
    str(UNET_BATCH_SIZE),
    "--det-threshold",
    str(DET_THRESHOLD),
    "--ilp-edge-weight",
    str(ILP_EDGE_WEIGHT),
    "--ilp-appearance-weight",
    str(ILP_APPEARANCE_WEIGHT),
    "--ilp-disappearance-weight",
    str(ILP_DISAPPEARANCE_WEIGHT),
    "--ilp-division-weight",
    str(ILP_DIVISION_WEIGHT),
]
if USE_ILP:
    cmd.append("--use-ilp")

print(" ".join(cmd))
subprocess.run(
    cmd,
    cwd=REPO_DIR,
    env={**os.environ, "PYTHONPATH": "src"},
    check=True,
)

## Submission

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

pred_dirs = sorted(Path(REPO_DIR, "predictions").glob(f"*/{METHOD}/split_0"))
assert pred_dirs, "No prediction directory found"
pred_dir = pred_dirs[0]

export_script = Path(REPO_DIR) / "scripts" / "geffs_to_csv.py"
if export_script.exists():
    subprocess.run(
        [
            sys.executable,
            str(export_script),
            "--in-dir",
            str(pred_dir),
            "--csv",
            OUTPUT_PATH,
        ],
        cwd=REPO_DIR,
        env={**os.environ, "PYTHONPATH": "src"},
        check=True,
    )
else:
    code = r"""
import sys
from pathlib import Path
import zarr
import csv

pred_dir = Path(sys.argv[1])
out_path = Path(sys.argv[2])
rows = []
for geff in sorted(pred_dir.glob("*.geff")):
    root = zarr.open(str(geff), mode="r")
    node_ids = root["nodes"]["ids"][:]
    t = root["nodes"]["props"]["t"]["values"][:]
    z = root["nodes"]["props"]["z"]["values"][:]
    y = root["nodes"]["props"]["y"]["values"][:]
    x = root["nodes"]["props"]["x"]["values"][:]
    if "solution" in root["nodes"]["props"]:
        keep = root["nodes"]["props"]["solution"]["values"][:].astype(bool)
    else:
        keep = None
    id_map = {}
    for i, node_id in enumerate(node_ids):
        if keep is not None and not keep[i]:
            continue
        id_map[int(node_id)] = int(node_id)
        rows.append([
            geff.stem, "node", int(node_id), int(t[i]),
            int(round(float(z[i]))), int(round(float(y[i]))), int(round(float(x[i]))),
            -1, -1,
        ])
    edges = root["edges"]["ids"][:]
    if "solution" in root["edges"]["props"]:
        edge_keep = root["edges"]["props"]["solution"]["values"][:].astype(bool)
    else:
        edge_keep = None
    for i, (source, target) in enumerate(edges):
        if edge_keep is not None and not edge_keep[i]:
            continue
        source, target = int(source), int(target)
        if source not in id_map or target not in id_map:
            continue
        rows.append([geff.stem, "edge", -1, -1, -1, -1, -1, source, target])

with out_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow([
        "id", "dataset", "row_type", "node_id", "t", "z", "y", "x",
        "source_id", "target_id",
    ])
    for index, row in enumerate(rows):
        writer.writerow([index, *row])
print(len(rows), out_path)
"""
    subprocess.run(
        [sys.executable, "-c", code, str(pred_dir), OUTPUT_PATH],
        check=True,
    )

Path(OUTPUT_PATH).exists(), Path(OUTPUT_PATH).stat().st_size

## Submission Checks

In [ ]:
import csv
from pathlib import Path

expected = set(test_stems)
datasets = set()
row_count = 0
with Path(OUTPUT_PATH).open(encoding="utf-8") as file:
    reader = csv.DictReader(file)
    assert reader.fieldnames[0] == "id"
    for expected_id, row in enumerate(reader):
        assert int(row["id"]) == expected_id
        datasets.add(row["dataset"])
        row_count += 1

assert datasets == expected
assert row_count > 0
row_count, Path(OUTPUT_PATH).stat().st_size